In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: jaspy
#     language: python
#     name: jaspy
# ---

# NB 05 Per-step loss matrix, CVPE and the Model Confidence Set

Input: the `fold_models` artifact of NB04.

* Per-step loss = squared error of E[U1|U2] and E[U2|U1] at that step (CVPE integrand).
* Seeds are averaged **on the loss** within a fold; folds are averaged with `np.nanmean` over the folds that test the step (repeated hold-out: some steps are tested twice, the rest once; the counts are printed in notebook 00). The CVPE of a model is the mean of that series over all usable steps. The spread across folds is descriptive, not a standard error.
* MCS at alpha = 0.10 and 0.05 (the levels the text leads with) and alpha = 0.25 as a sensitivity level (weak evidence; a low alpha retains more models, so 0.25 is the lenient end, not "strict"). Block lengths 1, b/2, b, 2b with b the Politis-White length of the full usable sample (notebook 00); the primary length is the Politis-White length of the loss differentials.
* A larger pool lowers the power of each elimination step and retains more models; this is stated next to the table.
* The last section holds the tail-dependence analysis on a threshold grid; it reads the models and gates above and saves the `tail_grid` artifact for notebook 06.
* Every table of the report is written once, as LaTeX and CSV, in the sections *Tables for the report* and *Tail dependence*; the sections before them only display. The PIT ablation, the per-category MCS, the pair heat map and the fold-level CVPE tables are not part of the report and are not computed.

## At a glance

* **Purpose:** compare the 76 candidates on one common test sample (CVPE), build the Model Confidence Set, and add robustness checks and the tail-dependence grid.
* **Inputs:** `fold_models` and `ces_scores` (NB04). **Outputs:** `comparison_and_mcs`, `ces_comparison`, `tail_grid`, the tables of the report in `Tables 05_comparison_and_mcs/` (LaTeX and CSV) and the figure `chi_grid_empirical.png`.
* **Next step:** continue to NB06 for the remaining figures.
* **Sections:** Load, Pool and exclusions, Per-step loss series, Block-length diagnostic, MCS, Sensitivity, CVPE intervals, Within-fold MCS, Tables for the report, Copula Energy Score, Tail dependence.

## Comparison roadmap: what each step answers

| Step | Rule used in this notebook | Question it answers |
|---|---|---|
| Eligibility | A candidate enters the pool only if its marginal-uniformity statistic D_max is within the gate threshold in every fold and seed; a missing diagnostic counts as a failure | Is the fitted model a valid copula for the whitened pseudo-observations? |
| Common test sample | Per-step loss on the same test steps of the four purged folds; neural models are averaged over their three seeds before comparison | Are all models scored on identical observations? |
| CVPE | Mean over test steps of (u1 - E[U1 given U2])^2 + (u2 - E[U2 given U1])^2, using 100-node Gauss-Legendre quadrature; lower is better | How well does the fitted dependence predict one margin from the other? |
| CES | Not part of this table. When computed it is reported in its own table with its own MCS, never pooled with CVPE | Does the fitted joint dependence distribution match the realised pairs? |
| Block-bootstrap interval | Stationary bootstrap of the loss series with the primary block length | How uncertain is each model's mean CVPE? |
| MCS | Hansen-Lunde-Nason range test with stationary bootstrap, alpha 0.10 and 0.05 reported, alpha 0.25 sensitivity only, block lengths 1, b/2, b, 2b (b = full-sample Politis-White length) | Which models cannot be separated from the best at this confidence level? Survivor status is not proof of equivalence |
| Acar GLRT | Separate panel (when computed). It tests whether dependence varies with divergence within one family | Does dependence change with divergence? It does not rank neural against other models |

## Imports and settings

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"; os.environ["NUMEXPR_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import plotting_core

from run_support import load_artifact, save_artifact, SMOKE   # SMOKE: production default; override with THESIS_SMOKE (see run_support.py)
import fold_pipeline as FP
import gate_config as GC
import reporting_tables
plotting_core.configure("05_comparison_and_mcs")
reporting_tables.configure("05_comparison_and_mcs")

N_BOOT = 200 if SMOKE else 5000     # MCS bootstrap replicates (5,000 in the thesis)
ALPHAS = (0.10, 0.05, 0.25)   # lead with 0.10 and 0.05; 0.25 is a sensitivity level
B_FULL = int(load_artifact('preprocessing')['b_full'])      # Politis-White length of the full usable sample (data driven)
BLOCK_LENS = tuple(sorted({1, max(1, B_FULL // 2), B_FULL, 2 * B_FULL}))   # sensitivity lengths

## Load

In [ ]:
fm = load_artifact("fold_models")
STORE, ARR, FOLDS = fm['store'], fm['arrays'], fm['folds']
PARAM, ACAR, NEURAL, NONPARAM = fm['param'], fm['acar'], fm['neural'], fm['nonparam']
print("artifact built with smoke settings:", fm.get('smoke'))
gt = STORE['gate_table']
print(f"candidates {len(gt)}; with a loss in every fold {len(STORE['models'])}; in pool after gates {int(gt.in_pool.sum())}")
assert len(gt) == GC.N_CANDIDATES, (len(gt), GC.N_CANDIDATES)

## Pool and exclusions

How many of the 49 pairs survive the gates, and which models drop out and why (the gate table lists all 76 candidates; it is written for the report in the section *Tables for the report*). Check this before reading any MCS.

In [ ]:
print(gt.groupby('group').in_pool.agg(candidates='size', in_pool='sum').to_string())
print("\nConditional networks: marginal KS is the only numerical validity gate.")
display(gt[['group', 'max_ks', 'threshold', 'in_pool']].sort_values(['group', 'in_pool', 'max_ks'], ascending=[True, False, True]).round(4))

## Gate disclosure: every candidate's D_max, pool size by threshold, calibration check

The gate threshold is fixed before the runs (reference check D_max <= 0.07, aligned per check type). Every candidate's largest KS distance is reported, together with the pool that would result at 0.05, 0.07 and 0.10. The calibration rule (threshold at least the simulated 99.9th percentile of the null) is checked here.

In [ ]:
CAL = GC.check_ks_calibration(raise_on_fail=False)
print(f"reference threshold {GC.KS_D_MAX}; simulated 99.9th percentile of the reference null {CAL['ref_null_q999']:.4f}; rule satisfied: {CAL['ok']}")
if not CAL['ok']:
    print(f"WARNING: raise KS_D_MAX to {CAL['recommended']:.3f} (pre-declared rule) before the production run")
POOL_SIZES = FP.pool_size_table(gt)
reporting_tables.show_table(POOL_SIZES, 'gate_pool_sizes', caption='Pool size by group at each reference threshold.')
NEURAL_ROWS = gt[(gt.group == 'Neural') | gt.index.isin([FP.UNCOND_CODINE, FP.SPLIT_CODINE])]
n_neural_in = int(NEURAL_ROWS.in_pool.sum())
print(f"\nNeural candidates (12) passing the gate: {n_neural_in}")
if n_neural_in < 2:
    print("FALLBACK (pre-declared): fewer than two neural candidates pass. Report the best neural candidate of each family "
          "below, flagged as failing the gate, and say so in the text; the MCS pool itself is not changed.")
    display(NEURAL_ROWS[['max_ks', 'threshold', 'in_pool']])

## Per-step loss series (the MCS input)

In [ ]:
L, MODELS = FP.per_step_loss(STORE, only_pool=True)
GROUP = {m: STORE['group'][m] for m in MODELS}
print("per-step loss matrix:", L.shape, "(steps x models in pool)")
cvpe = pd.Series(L.mean(axis=0), index=MODELS, name='CVPE').sort_values()

## Block-length diagnostic (decides the primary MCS block length)

A Politis-White block length estimated on the loss differentials of the best model against the others (median across pairs). Data-driven rule: the estimate alone, no floor and no fixed length anywhere. If block length 1 gives the same confidence sets as the longer blocks, say so in the text; if not, that is the evidence for the blocks.

In [ ]:
dd = FP.differential_diagnostics(L, MODELS, nlags=250)
print("best model:", dd['best'], "| median Politis-White block length on the differentials:", dd['block_length_median'])
PRIMARY_BLOCK = int(max(1, np.ceil(dd['block_length_median']))) if np.isfinite(dd['block_length_median']) else int(B_FULL)
print("primary block length (Politis-White on the loss differentials):", PRIMARY_BLOCK, "| full-sample b:", B_FULL)

## MCS at alpha 0.10, 0.05, 0.25 and block lengths 1, b/2, b, 2b

In [ ]:
BL_ALL = tuple(sorted(set(BLOCK_LENS) | {PRIMARY_BLOCK}))
MCS_TAB, MCS_RAW = FP.mcs_table(L, MODELS, block_lens=BL_ALL, alphas=ALPHAS, n_boot=N_BOOT, seed=0)
pb = PRIMARY_BLOCK
main = pd.DataFrame({'group': [GROUP[m] for m in MODELS], 'CVPE': cvpe.reindex(MODELS).values,
                     'p-value': MCS_TAB[f'p@b{pb}'],
                     'in MCS (alpha=0.10)': MCS_TAB[f'in@b{pb}@a0.1'],
                     'in MCS (alpha=0.05)': MCS_TAB[f'in@b{pb}@a0.05'],
                     'in MCS (alpha=0.25, sensitivity)': MCS_TAB[f'in@b{pb}@a0.25']}, index=MODELS).sort_values('CVPE')
print(f"primary block length {pb}; pool {len(MODELS)} models")
display(main.head(20).round(5))
print("\nSurvivors per alpha:", {a: int(MCS_TAB[f'in@b{pb}@a{a:g}'].sum()) for a in ALPHAS})
print("\nSurvivor counts across block lengths (rows: alpha, columns: block length):")
display(pd.DataFrame({bl: {a: int(MCS_TAB[f'in@b{bl}@a{a:g}'].sum()) for a in ALPHAS} for bl in BL_ALL}).rename_axis('alpha'))

### Sensitivity: MCS under the alternative gate thresholds

Same loss series, same block length, pools defined by D_max <= 0.10 (loose) and <= 0.05 (strict). The primary MCS uses the pool at 0.07.

In [ ]:
SENS_MCS = {}
for _d in (GC.KS_D_MAX_LOOSE, GC.KS_D_MAX_STRICT):
    _pool = [m for m in FP.pool_at_threshold(gt, _d) if m in STORE['models']]
    if len(_pool) < 2:
        print(f"D_max <= {_d:.2f}: pool of {len(_pool)} models, MCS not run"); continue
    _Ls, _ms = FP.per_step_loss(STORE, only_pool=False, models=_pool)
    _tab, _ = FP.mcs_table(_Ls, _ms, block_lens=(pb,), alphas=(0.10, 0.05), n_boot=N_BOOT, seed=0)
    SENS_MCS[_d] = _tab
    print(f"D_max <= {_d:.2f}: pool {len(_ms)}; survivors alpha 0.10: {int(_tab[f'in@b{pb}@a0.1'].sum())}, alpha 0.05: {int(_tab[f'in@b{pb}@a0.05'].sum())}")

### Sensitivity of the survivor set to the bootstrap seed (0, 1, 2)

In [ ]:
stab = FP.mcs_seed_stability(L, MODELS, pb, alphas=ALPHAS, seeds=(0, 1, 2), n_boot=N_BOOT)
stab['n'] = stab.survivors.map(len)
base = stab[stab.seed == 0].set_index('alpha').survivors.map(set)
stab['same_as_seed0'] = [set(s) == base[a] for s, a in zip(stab.survivors, stab.alpha)]
display(stab[['seed', 'alpha', 'n', 'same_as_seed0']])

## Robustness: MCS separately for each fold

Every fold scores a test step on its own conditional PIT, so the pooled loss series averages log scores on different scales (always the same folds for all models; the assertion in `per_step_loss` checks this). As a robustness check the MCS is also run on each fold's own test window: one PIT scale, nothing averaged, no stitching. This is possible for the folds whose test steps form one contiguous stretch (A1, A2, B1). B2 tests two separated outer quarters and is skipped, because joining them would put an artificial junction into the bootstrap. Agreement with the pooled MCS supports the pooled result.

In [ ]:
FOLD_MCS, FOLD_SKIPPED = {}, []
for _f in FOLDS:
    try:
        _Lf, _msf, _pos = FP.per_fold_loss(STORE, _f, models=MODELS, only_pool=False)
    except ValueError as _e:
        FOLD_SKIPPED.append(_f); print(f"fold {_f} skipped: {_e}"); continue
    _tab, _ = FP.mcs_table(_Lf, _msf, block_lens=(pb,), alphas=(0.10, 0.05), n_boot=N_BOOT, seed=0)
    FOLD_MCS[_f] = _tab
    print(f"fold {_f}: {len(_Lf)} test steps; survivors alpha 0.10: {int(_tab[f'in@b{pb}@a0.1'].sum())}, alpha 0.05: {int(_tab[f'in@b{pb}@a0.05'].sum())}")
BY_FOLD = pd.DataFrame({'group': [GROUP[m] for m in MODELS], 'p pooled': MCS_TAB[f'p@b{pb}'], 'in pooled': MCS_TAB[f'in@b{pb}@a0.1']}, index=MODELS)
for _f, _tab in FOLD_MCS.items():
    BY_FOLD[f'p {_f}'] = _tab[f'p@b{pb}']; BY_FOLD[f'in {_f}'] = _tab[f'in@b{pb}@a0.1']
_flags = [c for c in BY_FOLD.columns if c.startswith('in ')]
_sv = {c: set(BY_FOLD.index[BY_FOLD[c].astype(bool)]) for c in _flags}
def _jac(a, b):
    """Jaccard index of two sets (0 if both are empty).

    Parameters:
    a (set): First set.
    b (set): Second set.

    Returns:
    float: Size of the intersection divided by the size of the union.
    """
    return len(a & b) / max(len(a | b), 1)
print("survivors at alpha 0.10:", {c: len(v) for c, v in _sv.items()})
print("Jaccard with the pooled set:", {c: round(_jac(_sv['in pooled'], v), 2) for c, v in _sv.items() if c != 'in pooled'})
print("pooled survivors excluded in at least one fold:", sorted(_sv['in pooled'] - set.intersection(*_sv.values())))
_show = BY_FOLD[BY_FOLD[_flags].any(axis=1)].sort_values('p pooled', ascending=False)
_t = _show.drop(columns='group').copy(); _t.index.name = 'model'
for _c in _flags: _t[_c] = _t[_c].astype(bool)           # shade_bool only colours real booleans
display(_show.round(3))
reporting_tables.df_to_latex(_t, 'mcs_by_fold.tex',
    caption=(r'MCS $p$-values and membership at $\alpha=0.10$, pooled over all four folds and separately on the test window of each fold with a contiguous test set (A1, A2, B1). '
             r'Only models that survive in at least one analysis are shown. Green: model in the set; red: model eliminated.'),
    label='tab:mcs_by_fold', digits=3, longtable=True, shade_bool=True, compact=True,
    col_format=r'>{\raggedright\arraybackslash}p{0.38\textwidth}' + r'r>{\centering\arraybackslash}p{0.9cm}' * len(_flags),
    headers=['model'] + sum([[r'$p$ ' + _c[3:], 'in'] for _c in _flags], []),
    short_caption='MCS by fold, pooled and per test window')

## CVPE with block-bootstrap CIs on the averaged series (not a single fold)

In [ ]:
from model_evaluation import cvpe_ci_all_models
ci = cvpe_ci_all_models(L, MODELS, b=pb, n_boot=min(N_BOOT, 2000), show_progress=False)
ci['group'] = [GROUP[m] for m in ci.index]
display(ci.sort_values('mean_loss').head(15).round(5))

## Joint top-20% CVPE (descriptive, no CI)

The region is defined per fold from that fold's test pseudo-observations (the PIT is refit per fold); folds are combined with fold-share weights.

In [ ]:
top20 = FP.joint_top_cvpe(STORE, ARR, models=MODELS, q=0.80)
print("steps in the region per fold:", top20.attrs['n_per_fold'])
cmp = pd.DataFrame({'CVPE (pooled)': cvpe, 'CVPE (joint top-20%, fold-share weighted)': top20['weighted']}).sort_values('CVPE (joint top-20%, fold-share weighted)')
display(cmp.head(15).round(5))

## Save

In [ ]:
save_artifact("comparison_and_mcs",
              model_names=MODELS, loss_matrix=L, groups=GROUP, cvpe=cvpe, primary_block=pb, block_lens=BL_ALL,
              alphas=ALPHAS, mcs_table=MCS_TAB, main_table=main, mcs_raw=MCS_RAW, seed_stability=stab,
              cvpe_ci=ci, top20=top20, top20_compare=cmp, gate_table=gt, selected_pairs=STORE['selected_pair'],
              diff_diagnostics={k: v for k, v in dd.items() if k != 'acf_all'}, n_boot=N_BOOT, smoke=SMOKE)
print("saved comparison_and_mcs")

## Tables for the report

Written from the saved `comparison_and_mcs` and `fold_models` artifacts; nothing is recomputed and nothing is displayed (the tables were shown above). Every table is written as LaTeX and CSV. MCS wording: results lead with alpha = 0.10 and 0.05; alpha = 0.25 is a sensitivity level read as weak evidence. Files go to the folder `Tables 05_comparison_and_mcs`.

In [ ]:
import reporting_tables as RT
cmp_art = load_artifact("comparison_and_mcs"); fm = load_artifact("fold_models")
main, gt = cmp_art['main_table'], cmp_art['gate_table']
PB = cmp_art['primary_block']; FOLDS = fm['folds']
print("primary block", PB, "| smoke artifact:", cmp_art.get('smoke'), fm.get('smoke'))

## MCS tables (top 20 with the CVPE interval, all models in the pool) and group summary

In [ ]:
RT.mcs_fold_design_table(main, PB, filename='mcs_main_top20.tex', longtable=False, top=20, label='tab:mcs_top20', ci=cmp_art['cvpe_ci'])
RT.mcs_fold_design_table(main, PB, filename='mcs_appendix_longtable.tex', longtable=True, label='tab:mcs_appendix')
RT.mcs_group_summary_table(main, gt)

## Gate table and excluded models

In [ ]:
g = gt.copy(); g.index.name = 'model'
g['in_pool'] = g['in_pool'].astype(bool)          # shade_bool only colours real booleans
cols = [c for c in ('group', 'max_ks', 'threshold', 'in_pool') if c in g.columns]
RT.df_to_latex(g[cols].sort_values(['group', 'in_pool'], ascending=[True, False]), 'gate_table_longtable.tex',
               r'Marginal-uniformity screening for all 76 candidates. The threshold is $D_{\max}\leq 0.07$ for a check with two KS tests (one simulated copula, two margins), calibrated to the 99.9th percentile of the simulated null, and is raised by the simulated null margin for the 14-test checks (seven covariate percentiles, two margins; about 0.076 exact sampler, 0.078 grid sampler); the column threshold gives the value used. $D_{\max}$ is the maximum KS distance of independently generated copula margins over the required folds, seeds, regimes and covariate values. Missing diagnostics fail the screen. Passing does not establish joint goodness-of-fit. Green: candidate enters the pool; red: candidate excluded.',
               'tab:gates', longtable=True,
               col_format=r'>{\raggedright\arraybackslash}p{0.46\textwidth}lrr>{\centering\arraybackslash}p{1.1cm}',
               headers=['model', 'group', 'max KS', 'threshold', 'in pool'],
               shade_bool=True, compact=True,
               short_caption='Marginal-uniformity screening, all 76 candidates')

## Block-length and alpha sensitivity of the survivor counts

In [ ]:
bl = cmp_art['block_lens']; mt = cmp_art['mcs_table']
sens = pd.DataFrame({f"b={b}": {f"alpha={a}": int(mt[f'in@b{b}@a{a}'].sum()) for a in ('0.1', '0.05', '0.25')} for b in bl})
sens.index.name = 'level'
RT.df_to_latex(sens, 'mcs_block_alpha_sensitivity.tex', 'Number of models in the MCS by bootstrap block length and significance level. ' + RT.MCS_CAPTION_ALPHA,
               'tab:mcs_sens', digits=0, col_format='l' + 'r' * len(bl))

## Seed stability of the MCS

In [ ]:
st = cmp_art['seed_stability'].copy(); st['survivors'] = st.survivors.map(lambda s: len(s))
RT.df_to_latex(st[['seed', 'alpha', 'n', 'same_as_seed0']] if 'n' in st else st[['seed', 'alpha', 'survivors']], 'mcs_seed_stability.tex',
               'MCS survivor counts under different bootstrap seeds at the primary block length.', 'tab:mcs_seed', index=False, digits=2)

## CVPE intervals of the unconditional and binary-split models, and the joint top-20% comparison

In [ ]:
ci = cmp_art['cvpe_ci'].copy(); ci.index.name = 'model'
listing = ci[ci.index.str.startswith(('Unconditional: ', 'Divergence split: '))]
RT.df_to_latex(listing[['n', 'mean_loss', 'ci_lo', 'ci_hi']], 'cvpe_ci_unconditional_binary.tex',
               'CVPE with stationary-bootstrap intervals, unconditional and binary-split models (listing order; the continuous Acar curves are not in this table).',
               'tab:cvpe_ci_listing', longtable=True, digits=5, compact=True,
               col_format=r'>{\raggedright\arraybackslash}p{0.52\textwidth}rrrr',
               headers=['model', '$n$', 'CVPE', 'CI low', 'CI high'],
               short_caption='CVPE with stationary-bootstrap intervals, unconditional and binary-split models')
top = cmp_art['top20_compare'].copy(); top.index.name = 'model'
RT.df_to_latex(top.head(20), 'joint_top20_cvpe_fold.tex',
               r'CVPE pooled versus restricted to each fold joint top-20\% region (fold-share weighted; descriptive).', 'tab:top20',
               digits=5, compact=True, col_format=r'>{\raggedright\arraybackslash}p{0.55\textwidth}rr',
               headers=['model', 'CVPE (pooled)', r'CVPE (top-20\%)'],
               short_caption=r'CVPE pooled and in the joint top-20\% region')

## Copula Energy Score comparison (separate from CVPE; never pooled)

Own table, own MCS on the CES series, same fold design, same test steps, same stationary bootstrap and block lengths. Models without a CES are listed with the reason. The CES MCS is run only on pool models that have a CES in every fold; networks scored from fewer than three seeds are flagged. The score is deterministic quadrature (no Monte Carlo error); its numerical settings are shown. Survivor status is not proof of equivalence.

In [ ]:
_ces = load_artifact("ces_scores")
if not _ces:
    print("CES not computed: run the 'Copula Energy Score' section of notebook 04 first (needs the stored networks).")
else:
    CS = _ces['store_raw']
    _ms = [m for m in CS['models'] if CS['in_pool'].get(m)]
    _L = np.nanmean(CS['tensor'][:, [CS['models'].index(m) for m in _ms], :], axis=2)
    if not np.isfinite(_L).all():
        print("CES series has NaN steps for some pool models: those models are excluded from the CES MCS")
        _keep = np.isfinite(_L).all(axis=0); _ms = [m for m, k in zip(_ms, _keep) if k]; _L = _L[:, _keep]
    _tab, _ = FP.mcs_table(_L, _ms, block_lens=(pb,), alphas=ALPHAS, n_boot=N_BOOT, seed=0)
    CES_MAIN = _ces['table'].loc[_ms].copy()
    CES_MAIN['p-value'] = _tab[f'p@b{pb}']
    for a in ALPHAS:
        CES_MAIN[f'in CES-MCS (alpha={a:g})'] = _tab[f'in@b{pb}@a{a:g}']
    CES_MAIN = CES_MAIN.sort_values('ces')
    display(CES_MAIN.head(20).round(5))
    print(f"pool models with a CES: {len(_ms)} of {int(gt.in_pool.sum())}; independence CES {_ces['table'].attrs['independence_ces']:.5f}; "
          f"estimator: {_ces['table'].attrs['estimator']}")
    print("survivors:", {a: int(_tab[f'in@b{pb}@a{a:g}'].sum()) for a in ALPHAS})
    _few = CES_MAIN[CES_MAIN.seeds_used.fillna(3) < 3]
    if len(_few):
        print(f"WARNING: {len(_few)} neural models are scored from fewer than 3 seeds (weights of the other seeds were not stored): {list(_few.index)[:4]} ...")
    print("models without a CES:", {m: r for m, r in list(CS['no_ces'].items())[:6]}, "..." if len(CS['no_ces']) > 6 else "")
    save_artifact("ces_comparison", main_table=CES_MAIN, mcs_table=_tab, primary_block=pb, alphas=ALPHAS, n_boot=N_BOOT, smoke=SMOKE)

### CES table for the report (written only when the CES comparison above was computed)

In [ ]:
_cc = load_artifact('ces_comparison')
if not _cc:
    print("CES comparison not computed: no ces_table.tex written. (Does not affect the other tables.)")
else:
    _t = _cc['main_table'].head(20).copy(); _t.index.name = 'model'
    _cols = [c for c in ('group', 'ces', 'vs_independence', 'seeds_used', 'p-value', 'in CES-MCS (alpha=0.1)', 'in CES-MCS (alpha=0.05)') if c in _t.columns]
    RT.df_to_latex(_t[_cols], 'ces_table.tex', 'Copula Energy Score (raw joint score, lower is better), top 20 models with CES-MCS membership. Independence reference in the text.', 'tab:ces', digits=5)
    print("written ces_table.tex")

## Tail dependence on a threshold grid

This section needs only the objects above. The coefficient chi-hat(q) = P(U2 > q | U1 > q) at q = 0.95 is a finite-threshold co-exceedance probability, not the asymptotic chi = lim chi(q). Under independence chi(q) = 1 - q (0.05 at q = 0.95). Student-t and Clayton impose a non-zero limit by their form; Gaussian and Frank have lambda = 0 yet give chi(0.95) > 0.05 for any positive dependence. So q and the family assumption drive the conclusion; this notebook shows chi-hat(q) and Coles' chi-bar(q) on q in {0.80, 0.85, 0.90, 0.95} with the number of joint exceedances behind every cell, block-bootstrap bands (mean block length b, the full-sample Politis-White length; regime-aware for regime cells, at least 200 replicates), and the model-implied curve chi_m(q) = P_m(U1>q, U2>q)/(1-q) against it.

Descriptive tables use the full-sample kernel-conditional u (labelled descriptive, **not used for selection**) and the fold-training version as a stability check. Neither selects families.

In [ ]:
import model_diagnostics as TG
N_BOOT_TAIL = 200 if SMOKE else 500   # >= 200 for reported bands
TAIL_BLOCK = B_FULL   # full-sample Politis-White length
QS = TG.Q_THRESHOLDS

## Empirical chi-hat(q), chi-bar(q) with counts and bands

(a) Full-sample kernel-conditional u (descriptive). `n_joint` is the number of joint exceedances; `expected_joint_indep` is what independence would give.

In [ ]:
f0 = load_artifact(f"preprocessing_fold_{FOLDS[0]}")
u1_full, u2_full, high_full = TG.full_sample_regime_u(f0['precip_aligned'], f0['wind_aligned'], f0['div_aligned'], h=float(f0['kernel_h']))
GRID_FULL = TG.chi_grid_with_bands(u1_full, u2_full, high_full, QS, b=TAIL_BLOCK, n_boot=N_BOOT_TAIL)
cols = ['group', 'tail', 'q', 'n', 'n_joint', 'expected_joint_indep', 'chi', 'chi_lo', 'chi_hi', 'independence', 'chibar', 'chibar_lo', 'chibar_hi']
display(GRID_FULL[cols].round(3))

# LaTeX version for the appendix: 6 columns instead of 13 (estimate and band in one cell; counts as observed / expected)
def _f(x):
    """Format a number with three decimals and a typeset minus sign.

    Parameters:
    x (float): Number to format.

    Returns:
    str: The LaTeX-ready text.
    """
    return f'{x:.3f}'.replace('-', '$-$')
_c = GRID_FULL
_t = pd.DataFrame({'regime': _c.group, 'tail': _c['tail'], '$q$': _c.q.map(lambda x: f'{x:.2f}'),
                   'joint exceed. (obs / exp)': [f'{int(a)} / {b:.1f}' for a, b in zip(_c.n_joint, _c.expected_joint_indep)],
                   r'$\hat\chi(q)$ [95\% band]': [f'{_f(a)} [{_f(b)}, {_f(d)}]' for a, b, d in zip(_c.chi, _c.chi_lo, _c.chi_hi)],
                   r'$\bar\chi(q)$ [95\% band]': [f'{_f(a)} [{_f(b)}, {_f(d)}]' for a, b, d in zip(_c.chibar, _c.chibar_lo, _c.chibar_hi)]})
_n = GRID_FULL.groupby('group')['n'].first()
def _tex_n(n):
    """Format an integer with a thousands separator that LaTeX typesets without extra space.

    Parameters:
    n (int): Number to format.

    Returns:
    str: The text, for example '2{,}884'.
    """
    return f'{int(n):,}'.replace(',', '{,}')
_n_regimes = (f'${_tex_n(_n["low"])}$ in each divergence regime' if _n['low'] == _n['high']
              else f'${_tex_n(_n["low"])}$ low-divergence and ${_tex_n(_n["high"])}$ high-divergence')
reporting_tables.df_to_latex(_t, 'chi_grid_full_sample_overview.tex',
    rf'Tail dependence over the full sample ($n={_tex_n(_n["all"])}$ steps; {_n_regimes}) with block-bootstrap bands. '
    r'Joint exceedances are the observed count and the count expected under independence, $n(1-q)^2$; under independence $\hat\chi(q)=1-q$ and $\bar\chi(q)=0$.',
    'tab:chi_grid_full_sample_overview', index=False, shade_bool=False, compact=True, raw_text=True, headers=list(_t.columns),
    col_format='llrcrr', short_caption='Tail dependence over the full sample', csv_data=GRID_FULL)

Sensitivity of the chi bands to the bootstrap block length (b/2 and 2b with b the full-sample Politis-White length; the estimates are identical, only the bands change).

In [ ]:
GRID_SENS = []
for _bl in (max(1, B_FULL // 2), 2 * B_FULL):
    _g = TG.chi_grid_with_bands(u1_full, u2_full, high_full, QS, b=_bl, n_boot=N_BOOT_TAIL)
    _g['block'] = _bl; GRID_SENS.append(_g)
GRID_SENS = pd.concat(GRID_SENS, ignore_index=True)
display(GRID_SENS[(GRID_SENS.group == 'all') & (GRID_SENS['tail'] == 'upper') & (GRID_SENS.q == 0.95)][['block', 'chi', 'chi_lo', 'chi_hi']].round(3))
reporting_tables.write_table(GRID_SENS, 'chi_grid_full_sample_block_sensitivity', index=False, compact=True,
                             caption='Tail dependence over the full sample for two other bootstrap block lengths (b/2 and 2b); the estimates are identical, only the bands change.')

(b) Fold-training version (stability check): the same grid computed on each fold's training rows with that fold's kernel-conditional u.

In [ ]:
GRID_FOLD = []
for f in FOLDS:
    A_ = ARR[f]; tr = A_['train']
    g = TG.chi_grid_with_bands(A_['u1'], A_['u2'], A_['high'], QS, b=TAIL_BLOCK, n_boot=N_BOOT_TAIL, seed=1, sample_mask=tr)
    g['fold'] = f; GRID_FOLD.append(g)
GRID_FOLD = pd.concat(GRID_FOLD, ignore_index=True)
display(GRID_FOLD[(GRID_FOLD.group == 'all') & (GRID_FOLD['tail'] == 'upper')][['fold', 'q', 'n_joint', 'expected_joint_indep', 'chi', 'chi_lo', 'chi_hi']].round(3))
print("Note: bootstrap blocks are drawn from the full timeline and only the requested rows are kept, so no block spans the join of B1's two training segments.")
reporting_tables.write_table(GRID_FOLD, 'chi_grid_by_fold', index=False, compact=True,
                             caption='Tail dependence by fold: the same grid computed on each fold\'s training rows with that fold\'s kernel-conditional pseudo-observations.')

## Figure: chi-hat(q) with bands, independence line 1-q, counts

In [ ]:
import figure_recipes as FR
FR.chi_grid_empirical(GRID_FULL)

## Model-implied chi_m(q) against the data

Joint-exceedance probabilities per test step come from the stored NB04 results (`fold_models`) (parametric: `cop.cdf`; Acar: local theta(w_t); neural: midpoint integration of the density over [q,1]^2 on a w grid, interpolated). chi_m(q) is averaged over each fold's test steps, by regime, and over folds. The empirical companion plotted with these model-implied curves uses the same fixed denominator, estimated joint exceedance probability divided by (1-q), and its corresponding bootstrap band. It is not the sample conditional-exceedance ratio chi-hat displayed in the preceding plot. The full-sample estimates and the model-implied test-step averages also use different step sets, so the comparison remains descriptive, not a test.

In [ ]:
surv = main['in MCS (alpha=0.10)']
SURV = [m for m in surv.index if bool(surv[m])]
groups = GROUP
SHOW = ['Uncond (AIC-selected)', 'Binary (AIC-selected pair)']
for g in ('Acar', 'Nonparametric', 'Neural'):
    ms = [m for m in cvpe.index if groups[m] == g]
    if ms: SHOW.append(ms[0])
print("survivors at alpha 0.10:", len(SURV), "| shown in the figure:", SHOW)

P_BY_Q = {}
for q in FP.Q_GRID:
    base = FP.collect_exceed_probs(PARAM, ACAR, NEURAL, q, nonparam=NONPARAM)
    for f in FOLDS:
        fp = PARAM[f]; un = min(fp['aic_uncond'], key=fp['aic_uncond'].get)
        base[f]['Uncond (AIC-selected)'] = base[f][f"Unconditional: {un}"]
        base[f]['Binary (AIC-selected pair)'] = base[f][FP.display_binary_name(*fp['selected_pair'])]
    P_BY_Q[q] = base
MODELS_MI = [m for m in dict.fromkeys(SHOW + SURV) if all(m in P_BY_Q[0.90][f] for f in FOLDS)]
MI = TG.model_implied_grid(P_BY_Q, ARR, MODELS_MI)
LAM = TG.asymptotic_lambda_table(PARAM, FOLDS)
display(MI[(MI.regime == 'all') & (MI.model.isin(SHOW))].pivot(index='model', columns='q', values='chi_model').round(3))
print("\nasymptotic lambda_U of the fitted parametric families (mean over folds):")
display(LAM.groupby(['scope', 'family'])[['lambda_U', 'lambda_L']].mean().round(3))

## Save the tail grid and write its tables

In [ ]:
reporting_tables.write_table(MI, 'model_implied_chi', index=False, compact=True,
                             caption='Model-implied co-exceedance chi(q) of the models shown in the figures, by regime and exceedance level q.')
reporting_tables.write_table(LAM, 'asymptotic_lambda', index=False,
                             caption='Asymptotic tail coefficients of the fitted parametric families, by fold and scope.')
save_artifact("tail_grid", grid_full=GRID_FULL, grid_fold=GRID_FOLD, model_implied=MI, asymptotic_lambda=LAM,
              shown=SHOW, survivors_a010=SURV, n_boot=N_BOOT_TAIL, block=TAIL_BLOCK, smoke=SMOKE)
print("saved tail_grid")